# S4_6 — Checkpoint Export: Leaf-JEPA Encoder

**Leaf-JEPA IRP** | Stage 4 — Leaf JEPA Pretraining

Selects the best checkpoint (by LP val macro-F1), exports ONLY the target_encoder
state dict as the Leaf-JEPA encoder, and provides the exact line to update in
config_stage3.py.

This is the FINAL output of Stage 4. After running this notebook, Stage 4 is complete.


## Initialization

In [8]:
import sys, json
from pathlib import Path

import torch
import timm


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import set_seed, export_leaf_jepa_encoder, plot_pretrain_curves

set_seed(RANDOM_SEED)
LEAF_JEPA_DIR.mkdir(parents=True, exist_ok=True)


## Load Best checkpoint

In [9]:
lp_history_path = PRETRAIN_DIR / "lp_monitor_history.json"
assert lp_history_path.exists(), f"Run S4_3 first: {lp_history_path}"

with open(lp_history_path) as f:
    lp_history = json.load(f)

best = max(lp_history, key=lambda r: r["lp_val_macro_f1"])
best_epoch  = best["pretrain_epoch"]
best_lp_f1  = best["lp_val_macro_f1"]

print(f"Best LP val macro-F1: {best_lp_f1:.4f} at pretrain epoch {best_epoch}")

# Find the checkpoint file
# Look for 'best' tagged checkpoint first, then epoch checkpoint
ckpt_best_tagged = PRETRAIN_CKPT_DIR / f"epoch_{best_epoch:04d}_best.pth"
ckpt_epoch       = PRETRAIN_CKPT_DIR / f"epoch_{best_epoch:04d}.pth"
# Find closest saved checkpoint
all_ckpts = sorted(PRETRAIN_CKPT_DIR.glob("*.pth"))

if ckpt_best_tagged.exists():
    use_ckpt = ckpt_best_tagged
elif ckpt_epoch.exists():
    use_ckpt = ckpt_epoch
elif all_ckpts:
    # Use last saved checkpoint
    use_ckpt = all_ckpts[-1]
    print(f"⚠️  Exact epoch checkpoint not found. Using latest: {use_ckpt.name}")
else:
    raise FileNotFoundError(f"No checkpoints found in {PRETRAIN_CKPT_DIR}")

print(f"Loading checkpoint: {use_ckpt}")


Best LP val macro-F1: 0.8848 at pretrain epoch 25
Loading checkpoint: /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/stage4/epoch_0025_best.pth


## Load & Verify Target Encoder

In [17]:
# Load and verify target encoder
ckpt_data = torch.load(use_ckpt, map_location="cpu")

# Rebuild target encoder architecture
target_encoder = timm.create_model(
    VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool="", no_embed_class= True
)
target_encoder.load_state_dict(ckpt_data["target_encoder"])
target_encoder.eval()

# Quick verification: forward pass
device_cpu = torch.device("cpu")
x = torch.randn(2, 3, IMAGE_CROP, IMAGE_CROP)
with torch.no_grad():
    out = target_encoder.forward_features(x)
print(f"Target encoder forward pass: {out.shape}  ✅")
print(f"  Expected: (2, {(IMAGE_CROP//PATCH_SIZE)**2 + 1}, {VIT_EMBED_DIM})")


/tmp/ipykernel_272/3898543124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_data = torch.load(use_ckpt, map_location="cpu")


Target encoder forward pass: torch.Size([2, 257, 1280])  ✅
  Expected: (2, 257, 1280)


In [18]:
# Export Leaf-JEPA encoder
export_leaf_jepa_encoder(
    target_encoder = target_encoder,
    export_path    = LEAF_JEPA_CHECKPOINT,
    epoch          = best_epoch,
    lp_val_f1      = best_lp_f1,
    config_info    = {
        "model":          VIT_MODEL_NAME,
        "embed_dim":      VIT_EMBED_DIM,
        "patch_size":     PATCH_SIZE,
        "pretrain_epochs":PT_EPOCHS,
        "best_epoch":     best_epoch,
        "biased_masking": ENABLE_BIASED_MASKING,
        "bias_strength":  SALIENCY_BIAS_STRENGTH,
        "ema_tau_start":  EMA_TAU_START,
        "ema_tau_end":    EMA_TAU_END,
        "dataset":        "PlantVillage-train",
        "frozen_layers":  list(FROZEN_LAYERS),
    }
)



  ✅ Leaf-JEPA encoder exported → /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/leafjepa-vit-h14-best.pth
     Epoch: 25  |  LP Val Macro-F1: 0.8848

  Next: Update LEAF_JEPA_CHECKPOINT in config_stage3.py:
     LEAF_JEPA_CHECKPOINT = Path('/workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/leafjepa-vit-h14-best.pth')


In [20]:
# Verify exported checkpoint loads correctly
print("\nVerifying exported Leaf-JEPA encoder...")
exported = torch.load(LEAF_JEPA_CHECKPOINT, map_location="cpu")

verify_enc = timm.create_model(VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool="", no_embed_class= True)
missing, unexpected = verify_enc.load_state_dict(exported["target_encoder"], strict=False)
print(f"  Missing keys:    {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

x = torch.randn(1, 3, IMAGE_CROP, IMAGE_CROP)
with torch.no_grad():
    out = verify_enc.forward_features(x)
print(f"  Forward pass:    {out.shape}  ✅")
print(f"  LP Val Macro-F1: {exported['lp_val_macro_f1']:.4f}")
print(f"  Best epoch:      {exported['epoch']}")
print(f"  Biased masking:  {exported['config']['biased_masking']}")

print("\n" + "="*60)
print("  ✅ STAGE 4 COMPLETE")
print("="*60)
print("\n  Update config_stage3.py with this line:")
print(f"  LEAF_JEPA_CHECKPOINT = Path('{LEAF_JEPA_CHECKPOINT}')")
print()
print("  Then run:")
print("  1. B5_leafjepa_linear_probe.ipynb  (Stage 3 key result)")
print("  2. B6_cleanlab_and_summary.ipynb   (Stage 3 final summary)")
print("  3. Stage 5 PEFT experiments")



Verifying exported Leaf-JEPA encoder...


/tmp/ipykernel_272/2984011102.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  exported = torch.load(LEAF_JEPA_CHECKPOINT, map_location="cpu")


  Missing keys:    0
  Unexpected keys: 0
  Forward pass:    torch.Size([1, 257, 1280])  ✅
  LP Val Macro-F1: 0.8848
  Best epoch:      25
  Biased masking:  True

  ✅ STAGE 4 COMPLETE

  Update config_stage3.py with this line:
  LEAF_JEPA_CHECKPOINT = Path('/workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/leafjepa-vit-h14-best.pth')

  Then run:
  1. B5_leafjepa_linear_probe.ipynb  (Stage 3 key result)
  2. B6_cleanlab_and_summary.ipynb   (Stage 3 final summary)
  3. Stage 5 PEFT experiments


## Final Plots

In [21]:
# Final plots and dissertation summary
with open(PRETRAIN_DIR / "pretrain_history.json") as f:
    history = json.load(f)
with open(PRETRAIN_DIR / "lp_monitor_history.json") as f:
    lp_history_full = json.load(f)

plot_pretrain_curves(
    history, lp_history_full,
    save_path = FIGURES_DIR / "S4_pretrain_curves.png",
    title     = "Leaf-JEPA Domain Pretraining — Final Training Curves"
)

# Stage 4 summary JSON
summary = {
    "stage":              "Stage 4 — Leaf-JEPA Pretraining",
    "encoder":            VIT_MODEL_NAME,
    "pretrain_epochs":    PT_EPOCHS,
    "best_epoch":         best_epoch,
    "best_lp_val_f1":     best_lp_f1,
    "biased_masking":     ENABLE_BIASED_MASKING,
    "checkpoint_path":    str(LEAF_JEPA_CHECKPOINT),
    "frozen_layers":      list(FROZEN_LAYERS),
    "ema_tau_start":      EMA_TAU_START,
    "ema_tau_end":        EMA_TAU_END,
    "pred_depth":         PRED_DEPTH,
    "pred_dim":           PRED_EMBED_DIM,
}
with open(PRETRAIN_DIR / "stage4_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Stage 4 summary saved.")
print("\nAll Stage 4 outputs:")
for path in sorted(PRETRAIN_DIR.iterdir()):
    print(f"  {path.name}")


  Pretraining curves saved → /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/figures/S4_pretrain_curves.png
Stage 4 summary saved.

All Stage 4 outputs:
  S4_masking_ablation.json
  ablation_standard_lp_history.json
  lp_monitor_history.json
  pretrain_history.json
  stage4_summary.json
